In [1]:
from pathlib import Path
import pandas as pd
import zipfile

with zipfile.ZipFile("../../parkinsons_drawings.zip", "r") as z:
    z.extractall("parkinsons_drawings")

DATA_ROOT = Path("../parkinsons_drawings")

records = []
for split in ['training', 'testing']:
    for label_name in ['healthy', 'parkinson']:
        folder = DATA_ROOT / 'spiral' / split / label_name
        if not folder.exists():
            continue
        for img_path in folder.glob('*'):
            if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg', '.bmp']:
                records.append({
                    'path': str(img_path),
                    'label': 1 if label_name == 'parkinson' else 0,
                    'label_name': label_name,
                    'original_split': split
                })

df = pd.DataFrame(records)
print(f'Total images found: {len(df)}')
print(df['label_name'].value_counts())

Total images found: 102
label_name
healthy      51
parkinson    51
Name: count, dtype: int64


In [2]:
print(f"Всего изображений: {len(df)}\n")
for label in ['healthy', 'parkinson']:
    for split in ['training', 'testing']:
        subset = df[(df['label_name']==label) & (df['original_split']==split)]['path'].tolist()
        print(f"--- {label} / {split} ({len(subset)}) — первые 10 имён ---")
        for p in sorted(subset)[:10]:
            print(" ", Path(p).name)
        print()

Всего изображений: 102

--- healthy / training (36) — первые 10 имён ---
  V01HE02.png
  V01HE03.png
  V02HE02.png
  V02HE03.png
  V03HE2.png
  V03HE3.png
  V04HE02.png
  V04HE03.png
  V05HE02.png
  V05HE03.png

--- healthy / testing (15) — первые 10 имён ---
  V01HE01.png
  V02HE01.png
  V03HE1.png
  V04HE01.png
  V05HE01.png
  V06HE01.png
  V07HE01.png
  V08HE01.png
  V09HE01.png
  V10HE01.png

--- parkinson / training (36) — первые 10 имён ---
  V01PE02.png
  V01PE03.png
  V02PE02.png
  V02PE03.png
  V03PE02.png
  V03PE03.png
  V03PE05.png
  V03PE06.png
  V03PE08.png
  V03PE09.png

--- parkinson / testing (15) — первые 10 имён ---
  V01PE01.png
  V02PE01.png
  V03PE01.png
  V03PE04.png
  V03PE07.png
  V04PE01.png
  V05PE01.png
  V06PE01.png
  V07PE01.png
  V08PE01.png



In [3]:
import re
from pathlib import Path

def extract_subject(path):
    stem = Path(path).stem
    m = re.match(r'^(.*?)(\d+)$', stem)
    return m.group(1) if m else stem

df['subject_id'] = df['path'].apply(extract_subject)

# Сверка: внутри одной группы лейбл должен быть всегда одинаковым
assert (df.groupby('subject_id')['label'].nunique() == 1).all(), "Разные лейблы внутри одной группы!"

print(f"Всего изображений: {len(df)}")
print(f"Уникальных subject_id: {df['subject_id'].nunique()}\n")

per_subject = df.groupby('subject_id').agg(label=('label', 'first'), n=('label', 'size'))
print(per_subject['label'].map({0: 'healthy', 1: 'parkinson'}).value_counts())
print("\nЗаписей на субъекта:")
print(per_subject['n'].describe())
print("\nТоп-10 субъектов по числу записей:")
print(per_subject.sort_values('n', ascending=False).head(10))

Всего изображений: 102
Уникальных subject_id: 28

label
parkinson    15
healthy      13
Name: count, dtype: int64

Записей на субъекта:
count    28.000000
mean      3.642857
std       2.497618
min       3.000000
25%       3.000000
50%       3.000000
75%       3.000000
max      15.000000
Name: n, dtype: float64

Топ-10 субъектов по числу записей:
            label   n
subject_id           
V55HE           0  15
V03PE           1   9
V02HE           0   3
V01HE           0   3
V02PE           1   3
V03HE           0   3
V04HE           0   3
V04PE           1   3
V05HE           0   3
V05PE           1   3


In [4]:
for sid in ['V55HE', 'V03PE']:
    files = df[df['subject_id'] == sid]['path'].tolist()
    print(f"=== {sid} ({len(files)} файлов) ===")
    for f in sorted(files):
        print(" ", Path(f).name)
    print()

=== V55HE (15 файлов) ===
  V55HE12.png
  V55HE13.png
  V55HE14.png
  V55HE15.png
  V55HE01.png
  V55HE02.png
  V55HE03.png
  V55HE04.png
  V55HE05.png
  V55HE06.png
  V55HE07.png
  V55HE08.png
  V55HE09.png
  V55HE10.png
  V55HE11.png

=== V03PE (9 файлов) ===
  V03PE01.png
  V03PE04.png
  V03PE07.png
  V03PE02.png
  V03PE03.png
  V03PE05.png
  V03PE06.png
  V03PE08.png
  V03PE09.png



In [5]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              balanced_accuracy_score, roc_auc_score)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

# MobileNetV2 как feature extractor — как и для MRI, transfer-learned features
base_model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))

def extract_mobilenet_features(path):
    img = Image.open(path).convert('RGB').resize((224, 224))
    arr = preprocess_input(img_to_array(img))
    return base_model.predict(np.expand_dims(arr, 0), verbose=0).flatten()

print("Извлечение признаков MobileNetV2...")
features = np.stack([extract_mobilenet_features(p) for p in df['path']])
y = df['label'].values
groups = df['subject_id'].values
print(f"Признаки: {features.shape}")

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20, random_state=42)),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
                class_weight='balanced', random_state=42))
])

logo = LeaveOneGroupOut()
print(f"LOSO: {logo.get_n_splits(features, y, groups=groups)} фолдов")

rec_true, rec_pred, rec_prob, rec_subject = [], [], [], []
for train_idx, test_idx in logo.split(features, y, groups=groups):
    pipe.fit(features[train_idx], y[train_idx])
    rec_true.extend(y[test_idx])
    rec_pred.extend(pipe.predict(features[test_idx]))
    rec_prob.extend(pipe.predict_proba(features[test_idx])[:, 1])
    rec_subject.extend(groups[test_idx])

rec_true, rec_pred, rec_prob = map(np.array, (rec_true, rec_pred, rec_prob))

print("\n=== Recording-level (pooled out-of-fold, n=102) ===")
print(f"accuracy      : {accuracy_score(rec_true, rec_pred):.3f}")
print(f"f1            : {f1_score(rec_true, rec_pred):.3f}")
print(f"sensitivity   : {recall_score(rec_true, rec_pred):.3f}")
print(f"specificity   : {recall_score(rec_true, rec_pred, pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(rec_true, rec_pred):.3f}")
print(f"auc           : {roc_auc_score(rec_true, rec_prob):.3f}")

res = pd.DataFrame({'subject': rec_subject, 'true': rec_true, 'pred': rec_pred, 'prob': rec_prob})
subj = res.groupby('subject').agg(true=('true', 'first'),
                                   pred=('pred', lambda s: s.mode()[0]),
                                   prob=('prob', 'mean'))

print(f"\n=== Subject-level (majority vote, n={len(subj)}) ===")
print(f"accuracy      : {accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"f1            : {f1_score(subj['true'], subj['pred']):.3f}")
print(f"sensitivity   : {recall_score(subj['true'], subj['pred']):.3f}")
print(f"specificity   : {recall_score(subj['true'], subj['pred'], pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"auc           : {roc_auc_score(subj['true'], subj['prob']):.3f}")

Извлечение признаков MobileNetV2...
Признаки: (102, 1280)
LOSO: 28 фолдов

=== Recording-level (pooled out-of-fold, n=102) ===
accuracy      : 0.775
f1            : 0.736
sensitivity   : 0.627
specificity   : 0.922
balanced_acc  : 0.775
auc           : 0.797

=== Subject-level (majority vote, n=28) ===
accuracy      : 0.714
f1            : 0.667
sensitivity   : 0.533
specificity   : 0.923
balanced_acc  : 0.728
auc           : 0.795


In [6]:
subj.reset_index().rename(columns={'subject': 'subject_id', 'true': 'label'})[['subject_id','label','prob','pred']] \
    .to_csv('results_spiral.csv', index=False)
print("Сохранено: results_spiral.csv")

Сохранено: results_spiral.csv


In [9]:
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold, GridSearchCV
from sklearn.decomposition import PCA

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20, random_state=42)),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=42))
])

param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear']
}

logo = LeaveOneGroupOut()
rec_true, rec_pred, rec_prob, rec_subject, fold_best_params = [], [], [], [], []

for fold_i, (train_idx, test_idx) in enumerate(logo.split(features, y, groups=groups), 1):
    X_train, y_train, groups_train = features[train_idx], y[train_idx], groups[train_idx]

    inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='balanced_accuracy', n_jobs=-1)
    grid.fit(X_train, y_train, groups=groups_train)

    fold_best_params.append(grid.best_params_)
    rec_true.extend(y[test_idx])
    rec_pred.extend(grid.predict(features[test_idx]))
    rec_prob.extend(grid.predict_proba(features[test_idx])[:, 1])
    rec_subject.extend(groups[test_idx])
    print(f"Fold {fold_i}/28: best_params={grid.best_params_}")

rec_true, rec_pred, rec_prob = map(np.array, (rec_true, rec_pred, rec_prob))

print(f"\n=== Recording-level (pooled out-of-fold, n={len(rec_true)}) ===")
print(f"accuracy      : {accuracy_score(rec_true, rec_pred):.3f}")
print(f"f1            : {f1_score(rec_true, rec_pred):.3f}")
print(f"sensitivity   : {recall_score(rec_true, rec_pred):.3f}")
print(f"specificity   : {recall_score(rec_true, rec_pred, pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(rec_true, rec_pred):.3f}")
print(f"auc           : {roc_auc_score(rec_true, rec_prob):.3f}")

res = pd.DataFrame({'subject': rec_subject, 'true': rec_true, 'pred': rec_pred, 'prob': rec_prob})
subj = res.groupby('subject').agg(true=('true', 'first'),
                                   pred=('pred', lambda s: s.mode()[0]),
                                   prob=('prob', 'mean'))

print(f"\n=== Subject-level (majority vote, n={len(subj)}) ===")
print(f"accuracy      : {accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"f1            : {f1_score(subj['true'], subj['pred']):.3f}")
print(f"sensitivity   : {recall_score(subj['true'], subj['pred']):.3f}")
print(f"specificity   : {recall_score(subj['true'], subj['pred'], pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"auc           : {roc_auc_score(subj['true'], subj['prob']):.3f}")

params_df = pd.DataFrame(fold_best_params)
print("\n=== Частота выбора гиперпараметров по 28 LOSO-фолдам (для R2.3) ===")
for col in params_df.columns:
    print(f"\n{col}:"); print(params_df[col].value_counts())

subj.reset_index().rename(columns={'subject': 'subject_id', 'true': 'label'})[['subject_id','label','prob','pred']] \
    .to_csv('results_spiral.csv', index=False)
print("\nСохранено: results_spiral.csv (обновлено)")

Fold 1/28: best_params={'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 2/28: best_params={'svm__C': 0.01, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 3/28: best_params={'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 4/28: best_params={'svm__C': 0.01, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 5/28: best_params={'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 6/28: best_params={'svm__C': 1, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Fold 7/28: best_params={'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 8/28: best_params={'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Fold 9/28: best_params={'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Fold 10/28: best_params={'svm__C': 10, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Fold 11/28: best_params={'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Fold 12/28: best_params={'svm__C': 0.01, 'svm__gamma': 's